<a href="https://colab.research.google.com/github/arurion/Tools/blob/main/MIDI2Video_for_BlackMIDI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install mido

import mido
import cv2
import numpy as np
import subprocess
import os
import random
import colorsys
from tqdm.notebook import tqdm

# ==========================================
#  設定フォーム (ここを変更してください)
# ==========================================

# @markdown ### 📁 ファイル設定
midi_filename = "input.mid" # @param {type:"string"}
audio_filename = "input.wav" # @param {type:"string"}
output_filename = "output.mp4" # @param {type:"string"}

# @markdown ### 🎬 映像モード設定
render_mode = "Horizontal_RtoL" # @param ["Horizontal_RtoL", "Horizontal_LtoR", "Vertical_Falling", "Vertical_Rising"]
# @markdown - Horizontal_RtoL: 右から左（標準）
# @markdown - Horizontal_LtoR: 左から右
# @markdown - Vertical_Falling: 上から下（Synthesia風）
# @markdown - Vertical_Rising: 下から上（音ゲー風）

# @markdown ### 🎨 色・ビジュアル設定
color_mode = "Single" # @param ["Random", "Single", "Rainbow"]
# 修正箇所: type:"color" を type:"string" に変更しました
base_color_hex = "#7f4dff" # @param {type:"string"}
enable_velocity = True # @param {type:"boolean"}
# @markdown - enable_velocity: ベロシティ（音の強弱）を色の濃淡に反映する

# @markdown ### 🎹 表示オプション
show_piano = True # @param {type:"boolean"}
show_note_names = True # @param {type:"boolean"}
pitch_range = 2 # @param {type:"integer"}
time_window = 4.0 # @param {type:"number"}

# @markdown ### 📐 解像度・マージン設定
width = 1920 # @param {type:"integer"}
height = 1080 # @param {type:"integer"}
fps = 60 # @param {type:"integer"}

margin_top = 50 # @param {type:"integer"}
margin_bottom = 50 # @param {type:"integer"}
margin_left = 150 # @param {type:"integer"}
margin_right = 50 # @param {type:"integer"}


# ==========================================
#  処理ロジック (ここから下は変更不要)
# ==========================================

def hex_to_bgr(hex_col):
    hex_col = hex_col.lstrip('#')
    rgb = tuple(int(hex_col[i:i+2], 16) for i in (0, 2, 4))
    return (rgb[2], rgb[1], rgb[0])

def get_color_for_note(channel, note, velocity):
    # ベースカラー決定
    if color_mode == "Single":
        base_bgr = hex_to_bgr(base_color_hex)
    elif color_mode == "Rainbow":
        hue = (note % 12) / 12.0
        r, g, b = colorsys.hsv_to_rgb(hue, 0.8, 1.0)
        base_bgr = (int(b*255), int(g*255), int(r*255))
    else: # Random
        random.seed(channel * 100) # チャンネルごとに固定
        base_bgr = (random.randint(50,255), random.randint(50,255), random.randint(50,255))

    # ベロシティ反映
    if enable_velocity:
        factor = 0.3 + 0.7 * (velocity / 127.0)
        return (int(base_bgr[0]*factor), int(base_bgr[1]*factor), int(base_bgr[2]*factor))
    else:
        return base_bgr

def is_black(n): return (n % 12) in [1, 3, 6, 8, 10]
def get_name(n): return ["C","C#","D","D#","E","F","F#","G","G#","A","A#","B"][n%12] + str(n//12-1)

def run_renderer():
    # ファイルチェック
    if not os.path.exists(midi_filename):
        print(f"❌ エラー: MIDIファイル '{midi_filename}' が見つかりません。アップロードしてください。")
        return
    if not os.path.exists(audio_filename):
        print(f"❌ エラー: 音声ファイル '{audio_filename}' が見つかりません。アップロードしてください。")
        return

    print(f"🎵 MIDI読み込み中: {midi_filename} ...")
    mid = mido.MidiFile(midi_filename)

    # データ収集用
    bend_times = {}
    bend_values = {}
    for ch in range(16):
        bend_times[ch] = [0.0]
        bend_values[ch] = [0.0]

    notes = []
    abs_time = 0.0
    active_notes = {}
    min_note, max_note = 21, 108 # 88鍵盤範囲

    # MIDI解析
    for msg in mid:
        abs_time += msg.time

        if msg.type == 'pitchwheel':
            val = (msg.pitch - 8192) / 8192.0 * pitch_range
            bend_times[msg.channel].append(abs_time)
            bend_values[msg.channel].append(val)

        elif msg.type == 'note_on' and msg.velocity > 0:
            key = (msg.channel, msg.note)
            active_notes[key] = {'start': abs_time, 'vel': msg.velocity}

        elif (msg.type == 'note_off') or (msg.type == 'note_on' and msg.velocity == 0):
            key = (msg.channel, msg.note)
            if key in active_notes:
                d = active_notes.pop(key)
                if min_note <= msg.note <= max_note:
                    col = get_color_for_note(msg.channel, msg.note, d['vel'])
                    notes.append({
                        'note': msg.note, 'start': d['start'], 'end': abs_time,
                        'ch': msg.channel, 'vel': d['vel'], 'color': col
                    })

    duration = abs_time + 3.0
    total_frames = int(duration * fps)

    # 座標計算の準備
    is_vertical = "Vertical" in render_mode
    is_reverse = "LtoR" in render_mode or "Rising" in render_mode

    draw_w = width - margin_left - margin_right
    draw_h = height - margin_top - margin_bottom
    total_keys = max_note - min_note + 1

    if is_vertical:
        key_step = draw_w / total_keys
        px_per_sec = draw_h / time_window
        hit_line = (height - margin_bottom) if not is_reverse else margin_top
    else:
        key_step = draw_h / total_keys
        px_per_sec = draw_w / time_window
        hit_line = margin_left if not is_reverse else (width - margin_right)

    # 映像生成開始
    temp_video = "temp_video_render.avi"
    fourcc = cv2.VideoWriter_fourcc(*'MJPG')
    out = cv2.VideoWriter(temp_video, fourcc, fps, (width, height))

    print(f"🎬 映像レンダリング開始 (全 {total_frames} フレーム / {duration:.1f}秒)")
    print(f"   モード: {render_mode}, サイズ: {width}x{height}")

    for f in tqdm(range(total_frames), desc="Rendering"):
        now = f / fps
        img = np.full((height, width, 3), (20, 20, 20), dtype=np.uint8)

        vis_min = now - time_window
        vis_max = now + time_window

        # ノート描画
        for n in notes:
            if n['end'] < vis_min or n['start'] > vis_max: continue

            # 時間解像度を上げて滑らかに
            step_dt = 0.05
            t_points = np.arange(n['start'], n['end'] + 0.001, step_dt)
            if len(t_points) < 2: t_points = np.array([n['start'], n['end']])

            # ピッチベンド適用
            b_vals = np.interp(t_points, bend_times[n['ch']], bend_values[n['ch']])

            dt_arr = t_points - now
            base_idx = n['note'] - min_note

            if not is_vertical:
                keys_arr = margin_top + (max_note - (n['note'] + b_vals)) * key_step
            else:
                keys_arr = margin_left + (base_idx + b_vals) * key_step

            k_thick = key_step + 1

            if is_vertical:
                ys = hit_line - dt_arr * px_per_sec if not is_reverse else hit_line + dt_arr * px_per_sec
                pts_L = np.column_stack((keys_arr, ys)).astype(np.int32)
                pts_R = np.column_stack((keys_arr + k_thick, ys)).astype(np.int32)
                poly_pts = np.vstack((pts_L, pts_R[::-1]))
            else:
                xs = hit_line + dt_arr * px_per_sec if not is_reverse else hit_line - dt_arr * px_per_sec
                pts_T = np.column_stack((xs, keys_arr)).astype(np.int32)
                pts_B = np.column_stack((xs, keys_arr + k_thick)).astype(np.int32)
                poly_pts = np.vstack((pts_T, pts_B[::-1]))

            # 発音中の発光
            color = n['color']
            if n['start'] <= now < n['end']:
                color = tuple([min(255, c + 150) for c in color])

            cv2.fillPoly(img, [poly_pts], color)

        # マージン塗りつぶし
        cv2.rectangle(img, (0,0), (width, margin_top), (15,15,15), -1)
        cv2.rectangle(img, (0,height-margin_bottom), (width,height), (15,15,15), -1)
        cv2.rectangle(img, (0,0), (margin_left,height), (15,15,15), -1)
        cv2.rectangle(img, (width-margin_right,0), (width,height), (15,15,15), -1)

        # 判定線と鍵盤
        line_pos = int(hit_line)
        if is_vertical:
            cv2.line(img, (0, line_pos), (width, line_pos), (200,200,200), 1)
            if show_piano:
                kb_y = line_pos if not is_reverse else line_pos - margin_top
                kb_h = margin_bottom if not is_reverse else margin_top
                for i in range(total_keys):
                    note_num = min_note + i
                    kx = int(margin_left + i * key_step)
                    kw = int(key_step) + 1
                    ib = is_black(note_num)
                    c = (40,40,40) if ib else (230,230,230)
                    cv2.rectangle(img, (kx, kb_y), (kx+kw, kb_y+kb_h), c, -1)
                    # 黒鍵の縁取り
                    cv2.rectangle(img, (kx, kb_y), (kx+kw, kb_y+kb_h), (100,100,100), 1)
                    if show_note_names and not ib and (note_num%12==0):
                        cv2.putText(img, get_name(note_num), (kx, kb_y+kb_h-10), cv2.FONT_HERSHEY_SIMPLEX, 0.35, (0,0,0), 1)
        else:
            cv2.line(img, (line_pos, 0), (line_pos, height), (200,200,200), 1)
            if show_piano:
                kb_x = 0 if not is_reverse else line_pos
                kb_w = margin_left if not is_reverse else margin_right
                for i in range(total_keys):
                    note_num = max_note - i
                    ky = int(margin_top + i * key_step)
                    kh = int(key_step) + 1
                    ib = is_black(note_num)
                    c = (40,40,40) if ib else (230,230,230)
                    cv2.rectangle(img, (kb_x, ky), (kb_x+kb_w, ky+kh), c, -1)
                    cv2.rectangle(img, (kb_x, ky), (kb_x+kb_w, ky+kh), (100,100,100), 1)
                    if show_note_names and not ib and (note_num%12==0):
                        cv2.putText(img, get_name(note_num), (kb_x+5, ky+int(kh*0.7)), cv2.FONT_HERSHEY_SIMPLEX, 0.35, (0,0,0), 1)

        out.write(img)

    out.release()
    print("🎥 映像生成完了。音声と結合中...")

    # FFmpeg結合
    cmd = [
        'ffmpeg', '-y',
        '-i', temp_video,
        '-i', audio_filename,
        '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-preset', 'fast',
        '-c:a', 'aac', '-b:a', '192k',
        '-shortest',
        output_filename
    ]
    subprocess.run(cmd)

    if os.path.exists(temp_video):
        os.remove(temp_video)

    print(f"✅ すべて完了しました！: {output_filename}")
    print("左側のファイル一覧からダウンロードしてください。")

if __name__ == "__main__":
    run_renderer()

🎵 MIDI読み込み中: input.mid ...
🎬 映像レンダリング開始 (全 11241 フレーム / 187.4秒)
   モード: Horizontal_RtoL, サイズ: 1920x1080


Rendering:   0%|          | 0/11241 [00:00<?, ?it/s]

🎥 映像生成完了。音声と結合中...
✅ すべて完了しました！: output.mp4
左側のファイル一覧からダウンロードしてください。


In [ ]:
from google.colab import files
import os

# 設定した出力ファイル名（もし変数を変更していたらここも変えてください）
target_file = output_filename  # 上のセルで設定した変数を使います

if os.path.exists(target_file):
    print(f"📥 {target_file} のダウンロードを開始します...")
    files.download(target_file)
else:
    print(f"⚠️ ファイル {target_file} が見つかりません。")

📥 output.mp4 のダウンロードを開始します...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>